# main追加評価：通常条件の量拡大 ＋ A/Bsの空間拡張（Windows）
既存30kモデルを固定し、**①元条件の確認評価 → ②広い空間でのA/Bs比較**を一つのNotebookで実施します。新規学習はしません。

- ① **A・B75・B50・E**：既存確認bankから各N25形状、計100形状。学習3方式×3seed＋Eで **1,000 episode**。Eは各形状1回です。
- ② **A・B75・B50のみ**：低CPAの交差航路を後方へ延ばした**広い空間1条件だけ**。各N10形状で **360 episode**。0/20/40秒の条件比較はしません。

既定の合計は**1,360 episode**。試走は本実行で再利用するので追加分ではありません。
両者のbank・保存先・集計・zipを分離します。①は通常分布の確認、②は低CPAを多く含む空間拡張の追加ストレス評価です。相互に混ぜて平均しません。
①の比較方式・指標は開始時に固定します。結果を見て再調整したモデルは同じ確認結果に混在させません。

②では交差領域・ゴールを保ち、出発点を航路沿いに後方へ延ばします。距離の指定には「初速×40秒」を内部的に用いますが、時間別の実験群は作りません。実際の空間寸法と接近猶予は保存した初期条件・軌跡から確認できます。

この後に **③ A_main_v1.1の追加学習・E/A_v1.0/A_v1.1比較**まで続けます。既定は各seed Critic500 update＋追加5,000 online step（episode終端で停止）。3seed計15,000追加online stepで、凍結方策の収集は別です。v1.1の比較評価は初回294episodeです。
上から実行すれば全工程が進みます。試走だけならRUN_FULL=False、v1.1を除くならRUN_V11=Falseにしてください。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from tqdm.notebook import tqdm
from scripts.main_route_extension_runner import prepare, checkpoints, select_scenarios, evaluate_grid, status, export_results
from scripts import main_confirmation_runner as confirmation
from uav_safe_marl.evaluation import load_scenario_bank
from uav_safe_marl.runtime import runtime_manifest
SOURCE = ROOT / 'results' / 'main_additional_20261120'
OUTPUT = ROOT / 'results' / 'main_spatial_extension_single_20261127'
BANK_PER_N = 100
EVAL_PER_N = 10
COUNTS = (2, 4, 8, 16)
LEADS = (40.,)  # 広い空間1条件のみ。延長量の比較はしない
SEEDS = (0, 1, 2)
EVALUATION_DEVICE = 'cpu'
RUN_PROBE = True
RUN_FULL = True
RUN_EXPORT = True

# ①通常条件：既存の未使用確認bankを固定して量を拡大
CONFIRM_OUTPUT = ROOT / 'results' / 'main_confirmation_20261127'
CONFIRM_BANK_PER_N = 100
CONFIRM_EVAL_PER_N = 25
RUN_CONFIRMATION = True
# ②空間拡張（1条件）：以下の既存OUTPUT/BANK_PER_N/EVAL_PER_Nを使用
RUN_EXTENSION = True
# RUN_PROBE / RUN_FULL / RUN_EXPORTは両実験に共通

# ③ A_main_v1.1：Aだけの通常航法修正。v1.0の保存先とは独立。
from scripts import main_v11_runner as v11
import torch
RUN_V11 = True
V11_OUTPUT = ROOT / 'results' / 'main_v1_1_navigation_20261127'
V11_TARGET = 5000  # v1.1追加online step。延長は同じ保存先で10000等へ増やす。
V11_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
V11_DEV_PER_N = 2
V11_TEST_PER_N = 4
V11_SETTINGS = dict(critic_updates=500, train_single=8, train_benign_per_n=8,
                   dev_per_n=25, test_per_n=25, benign_counts=(2, 4), replay_capacity=4000)


## ① 通常条件：固定モデルの確認評価
元mainと同一の四方交通・初期間隔30m・予測horizon50秒・終了上限を使います。
SOURCE/banks/confirmation_test.jsonのhashを元実験と照合し、初期状態を変更せず使用します。CPAの小さい形状だけを選びません。

N別を主とし、全N等重み平均を併記します。初期CPA別の接近距離・接近率、介入時間/強度/Cost、QP可解性/fallback、到着時間/経路/timeout、通信を解析対象として固定します。Eの同一結果を3seed分の独立試行として数えません。

CONFIRM_EVAL_PER_Nだけを減らせば段階実行できます。初回は各N25形状、計100形状・1,000episodeです。元の分析48形状から独立形状数を約2倍に増やします。必要なら50、100へ増やして未完了分だけ追加できます。bank自体は各N100形状で固定し、結果を見て形状を選び直しません。

In [ ]:
if RUN_CONFIRMATION:
    confirm_experiment = confirmation.prepare(SOURCE, CONFIRM_OUTPUT, per_n=CONFIRM_BANK_PER_N, counts=COUNTS)
    confirm_ids = confirmation.select_scenarios(CONFIRM_OUTPUT, CONFIRM_EVAL_PER_N)
    print('通常条件の今回の評価量:', len(confirm_ids) * (3 * len(SEEDS) + 1), 'episode')
    print('保存先:', CONFIRM_OUTPUT)
    confirm_models = confirmation.checkpoints(CONFIRM_OUTPUT, SEEDS)
    for (method, seed), path in confirm_models.items(): print(f'{method}/{seed}: {path}')
    print('選択した評価デバイス:', EVALUATION_DEVICE)
    print(json.dumps(runtime_manifest(), ensure_ascii=False, indent=2))

### 通常条件の確認実走と本実行
最小/最大Nの共通2形状をseed0で4方式、計8case試走し、そのまま本実行に進みます。
case数と実step数をtqdm表示し、直列で1モデルずつ処理します。完了caseは本実行・再開時に再利用します。
RUN_FULL=Falseなら両実験とも試走だけ、RUN_EXTENSION=Falseなら通常条件だけを実行します。

In [ ]:
if RUN_CONFIRMATION:
    confirm_bank = load_scenario_bank(CONFIRM_OUTPUT / 'banks' / 'confirmation_test.json')
    confirm_probe_ids = [next(s['scenario_id'] for s in confirm_bank if s['scenario_id'] in set(confirm_ids) and s['actual_agent_count'] == n)
                         for n in sorted({min(COUNTS), max(COUNTS)})]
    if RUN_PROBE:
        confirm_probe = confirmation.evaluate_grid(CONFIRM_OUTPUT, seeds=[SEEDS[0]], device=EVALUATION_DEVICE,
                                                   scenario_ids=confirm_probe_ids, progress_factory=tqdm)
        print('通常条件・確認済みcase:', confirm_probe['completed_cases'])
    if RUN_FULL:
        confirm_result = confirmation.evaluate_grid(CONFIRM_OUTPUT, seeds=SEEDS, device=EVALUATION_DEVICE,
                                                    scenario_ids=confirm_ids, progress_factory=tqdm)
        print('通常条件の今回の完了:', confirm_result['requested_completed_cases'], '/', confirm_result['requested_cases'])

### 通常条件の途中集計・Mac持ち帰り
生NPZ・全指標・bank・固定解析方針・使用モデル・hashを保存します。途中停止後もこのセルだけで取り出せます。
通常条件と空間拡張は別zipです。こちらを保存した後、下の空間拡張へ進みます。

In [ ]:
if RUN_CONFIRMATION:
    confirm_partial = confirmation.status(CONFIRM_OUTPUT)
    print(json.dumps(confirm_partial, ensure_ascii=False, indent=2))
    if RUN_EXPORT:
        confirm_archive = confirmation.export_results(CONFIRM_OUTPUT, include_models=True)
        print('通常条件の持ち帰りzip:', confirm_archive)
        print(f'サイズ: {confirm_archive.stat().st_size / 1024**2:.1f} MiB')

## ② 空間拡張：広い空間1条件の共通bank
BANK_PER_N=100で各N100形状の拡張bankを固定します。実行はEVAL_PER_N=10の計40形状、3方式×3seedで **360episode**。
狭い元空間や中間の延長量は評価しません。全形状で同じ延長規則を適用した広い空間だけを比較します。

低CPAの遭遇を保つため、開始位置を初速×40秒分だけ後方へ延ばします。予測horizonは90秒、終了上限も長い航路に合わせる従来の拡張仕様です。これは空間拡張に付随する評価設定であり、時間別の実験群ではありません。

EVAL_PER_Nだけなら同じOUTPUTで評価量を増やせます。BANK_PER_N/延長量/COUNTS/生成条件を変える場合は別OUTPUTへ保存します。
旧0/20/40秒版と混ざらないよう、保存先をmain_spatial_extension_single_20261127に分けています。

In [ ]:
if RUN_EXTENSION:
    experiment = prepare(SOURCE, OUTPUT, per_n=BANK_PER_N, counts=COUNTS, leads=LEADS)
    print(f"元形状: {experiment['base_geometries']} / 延長後シナリオ: {experiment['scenarios']}")
    print(f"全量: {experiment['scenarios'] * 3 * len(SEEDS):,} episode / 最大step: {experiment['max_steps']:,}")
    selected_ids = select_scenarios(OUTPUT, EVAL_PER_N)
    print('今回の評価量:', len(selected_ids) * 3 * len(SEEDS), 'episode')
    print('生成試行数:', experiment['generation_draws'])
    print('保存先:', OUTPUT)


## 評価モデルとデバイス
既存の親実験A/B、追加実験B50、またはSOURCE/modelsの正規名モデルを利用します。不足時は必要なパスを表示して停止します。
CPU評価を明示しています。安全QPもCPUで動作します。CUDAへ変える場合は開始前に設定し、同じ保存先の途中で変更しないでください。


In [ ]:
if RUN_EXTENSION:
    models = checkpoints(OUTPUT, SEEDS)
    for (method, seed), path in models.items(): print(f'{method}/{seed}: {path}')
    print('選択した評価デバイス:', EVALUATION_DEVICE)
    print(json.dumps(runtime_manifest(), ensure_ascii=False, indent=2))


## 短い実走確認
最小/最大Nの共通2シナリオをseed0の3方式、計6caseで実行し、保存・所要時間を確認します。
モデルとhashを検証し、case進捗と飛行中の実step数を表示します。
完了caseは全量評価で再利用します。RUN_FULL=Falseにすれば確認だけで止まります。

In [ ]:
if RUN_EXTENSION:
    bank = load_scenario_bank(OUTPUT / 'banks' / 'route_extension.json')
    probe_ids = []
    for n in sorted({min(COUNTS), max(COUNTS)}):
        for lead in sorted({min(LEADS), max(LEADS)}):
            probe_ids.append(next(s['scenario_id'] for s in bank if s['actual_agent_count'] == n and s['route_extension_seconds'] == lead))
    if RUN_PROBE:
        probe = evaluate_grid(OUTPUT, seeds=[SEEDS[0]], device=EVALUATION_DEVICE, scenario_ids=probe_ids, progress_factory=tqdm)
        print('確認済みcase:', probe['completed_cases'])


## 全量評価
1条件ずつ直列で実行し、並列workerは使いません。学習・Replay保存は行いません。
中断時に飛行中だった1caseはやり直し、完了markerと生NPZが揃うcaseは再実行しません。


In [ ]:
if RUN_EXTENSION:
    if RUN_FULL:
        result = evaluate_grid(OUTPUT, seeds=SEEDS, device=EVALUATION_DEVICE, scenario_ids=selected_ids, progress_factory=tqdm)
        print('今回の完了:', result['requested_completed_cases'], '/', result['requested_cases'])
        print('全bankで完了:', result['completed_cases'], '/', result['expected_all_seeds_cases'])


## 途中集計・取り出し
このセルは途中停止後にも実行できます。全episodeのJSON/CSV、生NPZ、bank、初期条件、出典、モデルhashを保持します。
集計の違反率は記録点での判定です。後から区間内最近接・接近ペア分布・Actor急変・可解性・通信離脱も解析できます。
成功限定の経路比/完了時間は、timeoutを含むepisode時間と分けて集計します。


In [ ]:
if RUN_EXTENSION:
    partial = status(OUTPUT)
    print(json.dumps(partial, ensure_ascii=False, indent=2))


## Macへ持ち帰るzip
既定でA/B75/B50の評価用モデルも含めます（Replay・再開用checkpoint・runtime一時ファイルは除外）。
zip内のexport_manifest.jsonに全ファイルのhashとbyte数を記録します。保存先は評価フォルダの外です。
このセルだけ途中で実行すれば、その時点の完了結果を取り出せます。


In [ ]:
if RUN_EXTENSION:
    if RUN_EXPORT:
        archive = export_results(OUTPUT, include_models=True)
        print('持ち帰りzip:', archive)
        print(f'サイズ: {archive.stat().st_size / 1024**2:.1f} MiB')


# ③ main_v1.1：通常航法の修正と接近予防の保持

ここまでの通常条件評価と空間拡張は、固定した **main_v1.0** の追加評価です。
この後に扱う **main_v1.1はAだけを対象**とし、E・A_v1.0・A_v1.1を比較します。
この節の方針を、下の実行セルで実施します。RUN_V11=TrueかつRUN_FULL=Trueなら、v1.0評価に続いて収集・Critic修正・Actor追加学習・共通評価まで進みます。

## 検証したいこと

「通常航法の経験を補うことで、A_v1.0の接近予防を保ちながら、Eに対する不要な迂回・減速・上下動を減らせるか」を検証します。Referenceへの完全一致や残差ゼロそのものを目標にはせず、実際の到達効率と安全性で判断します。主な既存根拠はN=2の接近予防と介入負担低減であり、N=4以上の介入強度・QP可解性には課題が残っています。

## 分岐と学習データ

- A_v1.0の各seedの30k checkpointから分岐し、元ファイルは上書きしません。親checkpointのhash・λ・α・optimizer・追加step/update数を記録します。
- full_resumeにReplayがあれば継承します。model_onlyには旧Replayがないため、同じ再開とは扱いません。既存の経験をどう用意するかを開始前に確定します。
- 単機直線：同じ初期条件で残差ゼロのReference軌道とActor軌道を実収集します。Actor軌道の操作だけを置き換えた架空の遷移は作りません。
- 非接近複数機：全ペアの初期予測CPAが75m以上の形状を使います。Actorが新たに作った接近・Costも保持します。生成不能時に別の分布へ黙って置き換えません。
- 従来接近複数機：v1.0の経験と新しいActorの軌道を残し、既存回避能力の忘却を防ぎます。
- サンプル比率の初期案は単機25%・非接近25%・従来接近50%。勾配更新を単機・非接近・接近・接近の順に回し、同じbatch sizeで25/25/50%に固定します。episode数の比率ではありません。

## 更新順序と制約の意味

| 段階 | 単機 | 複数機 |
| --- | --- | --- |
| Critic修正：Actor・λ・αを固定 | Reward Criticとそのtargetだけ | Reward/Cost Criticとそれぞれのtarget |
| Actor追加学習 | 報酬側でActor更新 | 報酬＋既存CAL項でActor更新 |

λはseedごとのv1.0最終値で固定し、報酬・Cost・予算は維持します。ただし実効CAL係数はmax(0, λ+c×開始状態Cost推定超過)なので固定λでも変化します。λ=0でもCAL項が必ずゼロになるわけではありません。

学習Replayの混合比と、制約を評価する開始分布を分けます。CALの開始状態評価はv1.0の分布を維持する案とし、単機・容易な形状の追加だけで制約評価が薄まらないようにします。Actor段階のαはv1.0から継承し、従来の自動調整を維持する案です。

現行backendはCriticとtargetをまとめて更新するため、単機のCost更新除外やCriticのみの学習は、明示的な実装・不変性テストを行ってから使用します。

## 比較と判断

| 場面 | E / A_v1.0 / A_v1.1の比較 |
| --- | --- |
| 単機・非接近 | 到着時間、経路長、横・上下の逸脱、残差、Actorが作った接近 |
| 従来接近 | 初期CPA別の最近接距離と20m未満接近率、補正時間・強度、QP解なし/fallback、到達効率 |
| 学習診断 | 実測returnとCriticの整合、λ/実効CAL/α、データ源別の学習量 |

v1.0の接近予防を維持できたかを、効率の改善と同時に判定します。単に悪化の有意差が出なかったことを「維持」の証明にはせず、許容悪化幅と確認量を開始前に決めます。モデル選択に使ったbankは開発用として扱い、最終確認には別の未使用形状を固定します。

追加学習そのものの効果と、データ構成変更の効果まで分けて主張する場合は、同じ追加学習量・固定λで従来データだけを継続する対照も必要です。E/A_v1.0/A_v1.1のみなら、通常航法修正を含む追加学習全体の効果として報告します。

## 今回の実行仕様

初回はCritic500 update、追加5,000 online step/seed。旧Replayは必須にせず、固定A_v1.0で従来シナリオを再収集する分岐です。単機はReference/Actor各4形状、非接近はN2/4各1形状、接近は従来の全N各1形状を最初に収集します。非接近の学習/評価は初回N2/4だけ、接近は全Nを残します。v1.1の成果物にはversion/parent hash/stageを付け、v1.0と別保存先・別集計で管理します。既存の評価用zipへv1.1の重みを同名で混ぜません。


## 追加診断に基づく動機と評価軸（2026-11-27）

以下はユーザーから提示された追加分析の要約です。既存A/E対応比較の値と、今回提示された実運動・急変・単独飛行の診断を区別して記録します。後者の集計ファイル・検出条件の完全な出典は、実装時に対応付けます。

- 接近予防：全Nで未割引距離CostはA6.285/E8.680、純HOCBF補正秒/UAVはA0.988/E1.342。一方、割引純介入Costはほぼ同じ。N2の寄与が大きいが、N4/8/16の初期CPA<30mペアでも20m未満接近率はA25.31%/E38.89%。
- 可解性：Aは14episode・126機体tickで通常QP解なし、Eは0。Aの接近予防をQP可解性の改善と同一視しない。
- 実運動（提示された診断値）：速度差分による平均加速度ノルムA0.625/E0.161 m/s²、開始5秒後以降A0.579/E0.084、機体当たり二乗加速度積分A22.73/E5.75。これは操作負担の指標であり、消費エネルギーの実測ではない。
- 急変（提示された検出結果）：開始5秒後以降、0.1秒で加速度ベクトル変化>2 m/s²の頻度A0.84/E0.24回/(機体・分)。設定した針状スパイク検出では該当なし、段差型が観察された。Actor出力変更と安全処理切替を分け、検出閾値・持続時間を固定して再評価する。
- 回避不要の例（提示された事例）：N4の候補ゼロ・実距離90m超・HOCBF補正ゼロの機体で、Aは直線から最大28.1m逸脱し26.1秒、Eは直線で22.2秒。
- 単独区間（提示された診断値）：他機が到着済みかつ残距離30m以上で、Aの有効Actor補正1.76 m/s²、目標方向成分−0.47、実加速度0.620。純HOCBF補正は0。単独区間と単機episodeは別に集計する。
- pre-main補助診断：14,097stepの途中モデル、当時の実装/sourceを一致させた単独4方向で不要な曲線・高度変化が再現したとの報告。main_v1.0やpre-main最終モデルの確認としては扱わない。

v1.1では、到着効率に加え、実加速度・二乗加速度積分・横/高度逸脱・単独区間の負方向補正から、持続的な不要操作の改善を評価する。QP可解性/fallbackは安全性の維持確認として継続する。Actorのギザギザした操作・急変の平滑化は将来課題へ分け、v1.1の達成条件には含めない。

### 学習開始前の報酬整合性診断

現在の報酬は到達進捗・時間・残差ペナルティを持つが、実加速度二乗積分や加速度変化を直接最小化する目的ではない。したがって原因候補は、学習量不足・経験/価値推定の偏りに加え、現行報酬と望む巡航特性の不一致も残る。

同じ単機初期条件でReferenceとA_v1.0を実行し、実際の学習定義と同じ割引・報酬項でreturn差を確認する。Referenceのreturnが高いのにCriticが低く評価していれば価値修正の根拠になる。Referenceのreturn自体が低い場合は、経験追加だけで望む挙動を学ぶと保証しない。まず報酬を変えず検証し、必要な目的変更は別条件として識別する。

SACのReward Criticはエントロピーを含むsoft targetなので、Q値を生の報酬returnとそのまま同一視しない。同じ状態・継続方策での比較か、エントロピーを整合させた校正として評価する。Reference軌道全体のreturnを、そのままActor継続下のQ(s,0)の正解としない。


## main_v1.1の範囲の確定：平滑化は将来課題

main_v1.1は、A_v1.0の予防的回避を残し、単機・非接近でReference航法への不要な干渉を減らす追加学習とする。対象はAの既存3seedで、E・A_v1.0・A_v1.1を共通条件で比較する。B群/HBA/Fの追加学習は含めない。

Actorのギザギザ動作や段差型の操作変更は、滑らかな指令へ改善する余地がある。ただし平滑化は今後の課題として扱い、v1.1では新しい平滑化フィルタ・加速度変化ペナルティ・Actor更新周期変更を導入しない。持続的な不要補正の削減と、時間方向の滑らかさは別の問題であり、前者が改善しても後者が改善したとは主張しない。将来の平滑化では回避応答の遅れやHOCBFとの整合も検証する。

追加学習は、v1.0から分岐→3種の実軌道収集→Actor固定でCritic修正→混合データでActor追加学習→未使用形状で比較、の順とする。λはseedごとの最終値で固定し、報酬/Cost/予算は維持する。CALの開始分布もv1.0を維持する案。単機ではReward側だけ、複数機ではReward/Cost双方を更新し、必要な回避を忘れないよう従来接近データを残す。

合否は通常航法の到達効率・不要操作の改善と、接近場面の予防効果・最終離隔・QP可解性/fallbackの維持をセットで判断する。実加速度二乗積分は操作負担の代理指標であり、実消費エネルギーと同一視しない。学習量・混合比・評価量はNotebook先頭で固定する。今回は少量の分岐評価であり、許容幅を用いた統計的な性能維持の証明までは行わない。


## 狙いの妥当性を検討したメモ

[main_v1.1の狙いと追加学習方針の妥当性](../../docs/main_v1_1_rationale_20261127.md)に、経験追加の合理性、現在の報酬との整合、固定λの限界、Critic修正の位置づけ、3者比較で主張できる範囲、平滑化を将来課題とする理由を保存しています。

結論は「狙いは合理的。ただし経験・価値推定の修正が効くかを検証する仮説であり、現行目的と望む巡航の整合も先に調べる」。E/A_v1.0/A_v1.1では追加学習全体の効果を評価し、個別要因の因果効果は追加対照なしに断定しません。


## v1.1の準備とデバイス確認
v1.0の学習・制御engineを変更せず、専用処理で更新対象を制御します。報酬・Cost・予算・HOCBF・Actor周期・平滑化は変更しません。λはseed別の親最終値で固定し、αはCritic段階固定、Actor段階は従来の自動調整です。

巨大な旧Replayは転送不要です。旧モデルから重みとoptimizerを引き継ぎ、固定v1.0で経験を再収集します。これはv1.0の厳密な学習再開ではありません。

CALの開始状態は元のtraining bankだけから作り、単機・非接近の追加開始状態で薄めません。初回の非接近はN=2/4だけ、接近は全N。学習・development・confirmationは形状を分離します。
CUDAが使える場合は学習にCUDA、使えなければCPUとし、実際のデバイス名を下に表示します。

In [ ]:
if RUN_V11 and RUN_FULL:
    v11_experiment = v11.prepare(SOURCE, V11_OUTPUT, **V11_SETTINGS)
    print('version:', v11_experiment['version'], '/ parent:', v11_experiment['parent_version'])
    print('学習デバイス:', V11_DEVICE)
    if V11_DEVICE.startswith('cuda'): print('GPU:', torch.cuda.get_device_name(torch.cuda.current_device()))
    else: print('CUDAは選択されていません。CPUで学習します。')
    print('Critic修正:', V11_SETTINGS['critic_updates'], 'update/seed')
    print('追加online:', V11_TARGET, 'step/seed（episode終端まで）')
    print('非接近N:', v11_experiment['bank_spec']['benign_counts'])
    print('Replay:', v11_experiment['replay_strategy'])
    print('保存先:', V11_OUTPUT)

## v1.0とEで通常航法の報酬差を確認
独立developmentの小さな共通集合を先に評価します。単機のReferenceとActorの実際の割引環境報酬差はnavigation_reward_diagnostic.jsonに保存します。SACのsoft-Q校正とは区別します。
この診断に合わせて自動で報酬や予算を変更することはありません。

In [ ]:
if RUN_V11 and RUN_FULL:
    v11_before = v11.evaluate_grid(V11_OUTPUT, target=V11_TARGET, split='development',
                                  versions=('A_v1.0', 'E'), per_n=V11_DEV_PER_N,
                                  device=EVALUATION_DEVICE, progress_factory=tqdm)
    print('通常航法の報酬診断:', V11_OUTPUT / 'navigation_reward_diagnostic.json')

## 固定v1.0で収集 → Critic修正 → Actor追加学習
各seedを直列で実行します。収集episode、Critic update、追加online stepとepisode数を表示します。
単機更新ではCost Critic・target・ActorのCost項を除外し、複数機更新は既存SAC/CALと同じ式です。

完了stageは再利用します。中断時は最後の完全checkpointまで戻り、Replay・optimizer・乱数・固定開始状態・進捗をまとめて復元します。巨大Replayの保存は粗いepisode境界、Critic途中保存は小さい状態だけです。モデルのみの途中結果も保存します。
V11_TARGETを増やせば同じrunで追加できます。学習仕様やデバイスを変える場合は別V11_OUTPUTにしてください。
操作の平滑化は将来課題のままです。

In [ ]:
if RUN_V11 and RUN_FULL:
    v11_models = v11.train_grid(V11_OUTPUT, target=V11_TARGET, device=V11_DEVICE, progress_factory=tqdm)
    for seed, path in v11_models.items(): print(f'A_main_v1.1/{seed}: {path}')

## E・A_v1.0・A_v1.1の共通評価
追加学習後のdevelopmentと、別形状のconfirmationを評価します。Eは形状ごと1回、Aの両版は各3seed。いずれも平均方策で実行します。
各domain/Nのbankは25形状ずつ固定し、初回はdevelopment各2・confirmation各4形状。計294episodeで、先のA_v1.0/E評価は再利用します。

初期CPA別の20m未満接近、区間内10m未満違反、QP可解性、到着効率、実加速度・二乗積分・直線逸脱を保存します。少量の結果から統計的な性能維持を証明したとは扱いません。生NPZからさらに詳細な解析ができます。

In [ ]:
if RUN_V11 and RUN_FULL:
    v11_development = v11.evaluate_grid(V11_OUTPUT, target=V11_TARGET, split='development',
                                       per_n=V11_DEV_PER_N, device=EVALUATION_DEVICE, progress_factory=tqdm)
    v11_confirmation = v11.evaluate_grid(V11_OUTPUT, target=V11_TARGET, split='confirmation',
                                        per_n=V11_TEST_PER_N, device=EVALUATION_DEVICE, progress_factory=tqdm)
    print('v1.1比較・保存済みcase:', v11_confirmation['completed_evaluation_cases'])

## v1.1の途中集計と持ち帰り
このセルは途中停止後にも実行できます。v1.0とは別zipにし、A_v1.0/A_v1.1のモデル・bank・生NPZ・学習ログ・出典・hashを同梱します。Replayや巨大再開checkpointは含めません。
途中モデルしかない場合はmodel_latest.ptとして識別します。再開はWindowsの元保存先で行います。

In [ ]:
if RUN_V11 and RUN_FULL:
    v11_partial = v11.status(V11_OUTPUT)
    print(json.dumps(v11_partial, ensure_ascii=False, indent=2))
    if RUN_EXPORT:
        v11_archive = v11.export_results(V11_OUTPUT, target=V11_TARGET)
        print('v1.1の持ち帰りzip:', v11_archive)
        print(f'サイズ: {v11_archive.stat().st_size / 1024**2:.1f} MiB')